## Predict doublets for droplet based techniques - using our own model from second pass integraiton

Load library

In [41]:
# Path-related libraries
import os
from pyhere import here  # For reproducible relative paths

# Model-related libraries
from sklearn.preprocessing import LabelEncoder  # For encoding categorical variables

# AnnData and single-cell analysis libraries
import scanpy as sc       # For preprocessing and visualization of single-cell data
import anndata as ad      # For handling AnnData objects

# Numerical and tensor operations
import numpy as np        # For numerical computations and array manipulations
import pandas as pd
import torch              # Core PyTorch library for tensor operations
import torch.nn as nn     # Neural network components
from torch.optim import Adam  # Optimizer for training models
import torch.nn.functional as F
from torch.distributions import Distribution, Gamma
from torch.distributions import Poisson as PoissonTorch
import warnings
from typing import Optional, Union
from torch.distributions.utils import (
    broadcast_all,
    lazy_property,
    logits_to_probs,
    probs_to_logits,
)
import copy               # For deep copying model states

Functions for model

In [2]:
def log_nb_positive(
    x: Union[torch.Tensor],
    mu: Union[torch.Tensor],
    theta: Union[torch.Tensor],
    eps: float = 1e-8,
    log_fn: callable = torch.log,
    lgamma_fn: callable = torch.lgamma,
):
    log = log_fn
    lgamma = lgamma_fn
    log_theta_mu_eps = log(theta + mu + eps)
    res = (
        theta * (log(theta + eps) - log_theta_mu_eps)
        + x * (log(mu + eps) - log_theta_mu_eps)
        + lgamma(x + theta)
        - lgamma(theta)
        - lgamma(x + 1)
    )

    return res

def _convert_mean_disp_to_counts_logits(mu, theta, eps=1e-6):
    if not (mu is None) == (theta is None):
        raise ValueError(
            "If using the mu/theta NB parameterization, both parameters must be specified"
        )
    logits = (mu + eps).log() - (theta + eps).log()
    total_count = theta
    return total_count, logits


def _convert_counts_logits_to_mean_disp(total_count, logits):
    theta = total_count
    mu = logits.exp() * theta
    return mu, theta


def _gamma(theta, mu):
    concentration = theta
    rate = theta / mu
    # Important remark: Gamma is parametrized by the rate = 1/scale!
    gamma_d = Gamma(concentration=concentration, rate=rate)
    return gamma_d

class NegativeBinomial(Distribution):
    def __init__(
        self,
        total_count: Optional[torch.Tensor] = None,
        probs: Optional[torch.Tensor] = None,
        logits: Optional[torch.Tensor] = None,
        mu: Optional[torch.Tensor] = None,
        theta: Optional[torch.Tensor] = None,
        scale: Optional[torch.Tensor] = None,
        validate_args: bool = False,
    ):
        self._eps = 1e-8
        if (mu is None) == (total_count is None):
            raise ValueError(
                "Please use one of the two possible parameterizations. Refer to the documentation for more information."
            )

        using_param_1 = total_count is not None and (
            logits is not None or probs is not None
        )
        if using_param_1:
            logits = logits if logits is not None else probs_to_logits(probs)
            total_count = total_count.type_as(logits)
            total_count, logits = broadcast_all(total_count, logits)
            mu, theta = _convert_counts_logits_to_mean_disp(total_count, logits)
        else:
            mu, theta = broadcast_all(mu, theta)
        self.mu = mu
        self.theta = theta
        self.scale = scale
        super().__init__(validate_args=validate_args)

    @property
    def mean(self):
        return self.mu

    @property
    def variance(self):
        return self.mean + (self.mean**2) / self.theta

    @torch.inference_mode()
    def sample(
        self,
        sample_shape: Optional[Union[torch.Size, tuple]] = None,
    ) -> torch.Tensor:
        """Sample from the distribution."""
        sample_shape = sample_shape or torch.Size()
        gamma_d = self._gamma()
        p_means = gamma_d.sample(sample_shape)

        # Clamping as distributions objects can have buggy behaviors when
        # their parameters are too high
        l_train = torch.clamp(p_means, max=1e8)
        counts = PoissonTorch(
            l_train
        ).sample()  # Shape : (n_samples, n_cells_batch, n_vars)
        return counts

    def log_prob(self, value: torch.Tensor) -> torch.Tensor:
        return log_nb_positive(value, mu=self.mu, theta=self.theta, eps=self._eps)

    def _gamma(self):
        return _gamma(self.theta, self.mu)

In [3]:
## Encoder
class Encoder(nn.Module):
    def __init__(
        self,
        input_dim: int,
        layer_sizes: list,
        embedding_classes: list,
        embedding_dims: list,
        use_batch_norm: bool = True,
        use_layer_norm: bool = False,
        encoder_bias: bool = True,
        activation_function: nn.Module = nn.LeakyReLU(),
        latent_size: int = 25,
        dropout_rate: float = 0.1,    
     ):
        super().__init__()

        ## Encoder
        encoder_layers = [input_dim + np.sum(embedding_dims)] + layer_sizes
        self.encoder = nn.ModuleList()
        for i in range(len(encoder_layers)-1):
            self.encoder.append(nn.Linear(encoder_layers[i], encoder_layers[i+1], bias = encoder_bias)),
            self.encoder.append(nn.BatchNorm1d(encoder_layers[i+1], affine=True) if use_batch_norm else nn.Identity())
            self.encoder.append(nn.LayerNorm(encoder_layers[i+1], elementwise_affine=False) if use_layer_norm else nn.Identity())
            self.encoder.append(activation_function)
            self.encoder.append(nn.Dropout(p=dropout_rate) if dropout_rate > 0 else nn.Identity())

        # Mean and variance encoder
        self.mean_encoder = nn.Linear(encoder_layers[-1], latent_size)
        self.var_encoder = nn.Linear(encoder_layers[-1], latent_size)
    
    def forward(self, x: torch.Tensor, e: list):
        x = torch.cat([x, e], dim = -1)
        for i in range(len(self.encoder)):
            x = self.encoder[i](x)
        mu = self.mean_encoder(x)
        log_var = self.var_encoder(x)
        return mu, log_var

In [4]:
## Decoder
class Decoder(nn.Module):
    def __init__(
        self,
        input_dim: int,
        layer_sizes: list,
        embedding_classes: list,
        embedding_dims: list,
        use_batch_norm: bool = True,
        use_layer_norm: bool = False,
        decoder_bias: bool = True,
        activation_function: nn.Module = nn.LeakyReLU(),
        latent_size: int = 25,
        dropout_rate: float = 0.1,     
     ):
        super().__init__()

        ## Decoder
        decoder_layers = [latent_size + np.sum(embedding_dims)] + layer_sizes + [input_dim]
        self.decoder = nn.ModuleList()
        for i in range(len(decoder_layers)-1):
            self.decoder.append(nn.Linear(decoder_layers[i], decoder_layers[i+1], bias = decoder_bias)),
            self.decoder.append(nn.BatchNorm1d(decoder_layers[i+1], affine=True) if use_batch_norm & (i <= (len(decoder_layers)-3)) else nn.Identity())
            self.decoder.append(nn.LayerNorm(decoder_layers[i+1], elementwise_affine=False) if use_layer_norm & (i <= (len(decoder_layers)-3)) else nn.Identity())
            self.decoder.append(activation_function if  (i <= (len(decoder_layers)-3)) else nn.Identity())
            self.decoder.append(nn.Dropout(p=dropout_rate) if (dropout_rate > 0) & (i <= (len(decoder_layers)-3)) else nn.Identity())
            #self.decoder.append(nn.Softmax(dim=-1) if i > (len(decoder_layers)-3) else nn.Identity())
    
    def forward(self, x: torch.Tensor, e: list):
        x = torch.cat([x, e], dim = -1)
        for i in range(len(self.decoder)):
            x = self.decoder[i](x)
        return x

In [5]:
## Module
class VAE(nn.Module):
    def __init__(
        self,
        input_dim: int,
        layer_sizes: list,
        embedding_classes: list,
        embedding_dims: list,
        latent_size: int = 25,
        use_batch_norm: bool = True,
        use_layer_norm: bool = False,
        encoder_bias: bool = True,
        decoder_bias: bool = True,
        dropout_rate: float = 0.1,
        activation_function: nn.Module = nn.LeakyReLU(),
    ):
        super().__init__()
        self.dropout_rate = dropout_rate
        
        ## Encoders
        self.encoder = Encoder(input_dim = input_dim, layer_sizes = layer_sizes, embedding_classes = embedding_classes, embedding_dims = embedding_dims, use_batch_norm = use_batch_norm, use_layer_norm = use_layer_norm, encoder_bias = encoder_bias, activation_function = activation_function, latent_size = latent_size, dropout_rate = dropout_rate)

        ## Decoder 
        self.decoder = Decoder(input_dim = input_dim, layer_sizes = layer_sizes, embedding_classes = embedding_classes, embedding_dims = embedding_dims, use_batch_norm = use_batch_norm, use_layer_norm = use_layer_norm, decoder_bias = decoder_bias, activation_function = activation_function, latent_size = latent_size, dropout_rate = dropout_rate)
            
        ## Embedings and conditional
        self.embeddings = nn.ModuleList()
        for i in range(len(embedding_classes)):
            self.embeddings.append(nn.Embedding(embedding_classes[i], embedding_dims[i], max_norm=1))

        ## Distribution parameters
        self.theta = torch.nn.Parameter(torch.randn(input_dim))

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        z = mu + eps * std
        return z

    def forward(self, x: torch.Tensor, study: torch.Tensor, library: torch.Tensor, source: torch.tensor, donor: torch.tensor, use_attention = False, pretrain = False):
        ## Setup
        x_ = x
        x = torch.log1p(x)

        ## Embeddings
        e_source = self.embeddings[0](source)
        e_library = self.embeddings[1](library)
        e_study = self.embeddings[2](study)
        e_donor = self.embeddings[3](donor)
        embeddings = torch.cat([e_source, e_library, e_study, e_donor], dim = -1)
        
        ## Encoding
        mu, log_var = self.encoder(x, embeddings)

        ## Reparameterization trick
        if self.training:
            z = self.reparameterize(mu, log_var)
        else:
            z = mu

        ## Decode
        xhat = F.softmax(self.decoder(z, embeddings), dim=1)
       
        # Distribution parameters
        rate = torch.exp(torch.log(x_.sum(1)).unsqueeze(1)) * xhat
        px = NegativeBinomial(mu=rate, theta=torch.exp(self.theta), scale=xhat)

        return px, rate, xhat, z, log_var, embeddings
        

Parameters

In [6]:
# Paths
base_dir = str(here('data/integrate/second_pass/'))
files_dir = os.path.join(base_dir, 'files') 
plot_dir = os.path.join(base_dir, 'plot') 
objects_dir = os.path.join(base_dir, 'objects') 

In [9]:
# To load model
hvg = 1000
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # Use GPU if available
batch_size = 256

# Setup a model, load weights and set into eval mode
model = VAE(hvg, [256,128], [2, 11, 22, 270], [4, 4, 8, 32])
model.load_state_dict(torch.load("/work/scRNAseq/JM/model_weights_2.pth"))
model.to(device)
model.eval()

VAE(
  (encoder): Encoder(
    (encoder): ModuleList(
      (0): Linear(in_features=1048, out_features=256, bias=True)
      (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): Identity()
      (3): LeakyReLU(negative_slope=0.01)
      (4): Dropout(p=0.1, inplace=False)
      (5): Linear(in_features=256, out_features=128, bias=True)
      (6): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (7): Identity()
      (8): LeakyReLU(negative_slope=0.01)
      (9): Dropout(p=0.1, inplace=False)
    )
    (mean_encoder): Linear(in_features=128, out_features=25, bias=True)
    (var_encoder): Linear(in_features=128, out_features=25, bias=True)
  )
  (decoder): Decoder(
    (decoder): ModuleList(
      (0): Linear(in_features=73, out_features=256, bias=True)
      (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): Identity()
      (3): LeakyReLU(negative_slope=0.01)
  

Load full Anndata Object

In [10]:
adata = ad.read_h5ad(os.path.join(objects_dir, 'AB_adata_full_with_latent.h5ad'))

Set up covariates and inserts

In [12]:
source_encoder = LabelEncoder()
adata.obs['cell_nuclei_encoded'] = source_encoder.fit_transform(adata.obs.cell_nuclei)
library_encoder = LabelEncoder()
adata.obs['library_prep_encoded']  = library_encoder.fit_transform(adata.obs.library_prep)
study_encoder = LabelEncoder()
adata.obs['study_encoded']  = study_encoder.fit_transform(adata.obs.ic_id_study)
donor_encoder = LabelEncoder()
adata.obs['donor_encoded']  = donor_encoder.fit_transform(adata.obs.ic_id_donor_overall)

Select highly variable genes

In [22]:
sc.pp.highly_variable_genes(adata, n_top_genes=hvg, batch_key="ic_id_study", subset = True)

# Save HVGs
hvg_list = adata.var[adata.var['highly_variable']].index.tolist()
hvg_save = os.path.join(files_dir, f'hvgs_{hvg}.txt')
with open(hvg_save, 'w') as f:
    for gene in hvg_list:
        f.write(gene + '\n')
print(f'Saved HVGs to {hvg_save}')

# Save object
adata.write(os.path.join(objects_dir, 'AB_adata_hvg.h5ad'))

Saved HVGs to /work/islet_cartography_scrna/data/integrate/second_pass/files/hvgs_1000.txt


Reload AnnData object, only with hvgs

In [23]:
adata = ad.read_h5ad(os.path.join(objects_dir, 'AB_adata_hvg.h5ad'))

Function to classify doublets

In [33]:
## Function to classify doubets
## Implementation of SOLO
import copy
def classify_doublets(model, adata_sample, doublet_ratio = 1, seed = 42, batch_size = 256, n_hidden = 128, epochs = 1000, lr = 0.0001, device = 'cuda', train_fraction = 0.8, patience = 20, verbose = True):
    
    # Create doublets
    if verbose:
        print("Simulating doublets.")
    n_obs = adata_sample.n_obs
    num_doublets = doublet_ratio * n_obs
    random_state = np.random.RandomState(seed=seed)
    parent_inds = random_state.choice(n_obs, size=(num_doublets, 2))
    doublets = adata_sample.layers['counts'][parent_inds[:, 0]] + adata_sample.layers['counts'][parent_inds[:, 1]]
    
    # Get latent of doublets
    if verbose:
        print("Encoding simulated doublets.")
    model.eval()
    indices = np.arange(num_doublets)
    doublet_latent = []
    for i in range(0, num_doublets, batch_size):    
        # Get training data for current mini-batch        
        batch = indices[i:i + batch_size]
        train_data = torch.Tensor(doublets[batch,:].toarray()).to(device)
        train_library = torch.Tensor(np.unique(adata_sample.obs['library_prep_encoded'])).long().repeat(train_data.shape[0]).to(device)
        train_source = torch.Tensor(np.unique(adata_sample.obs['cell_nuclei_encoded'])).long().repeat(train_data.shape[0]).to(device)
        train_study = torch.Tensor(np.unique(adata_sample.obs['study_encoded'])).long().repeat(train_data.shape[0]).to(device)
        train_donor = torch.Tensor(np.unique(adata_sample.obs['donor_encoded'])).long().repeat(train_data.shape[0]).to(device)
    
        px, rate, xhat, z, log_var, embeddings = model(x = train_data, donor = train_donor, study = train_study, library = train_library, source = train_source)
        doublet_latent.append(z.detach().cpu())
    doublet_latent = torch.concatenate(doublet_latent)
    doublet_label = torch.tensor(1).repeat(doublet_latent.shape[0])
    
    # Get latent of real cells
    if verbose:
        print("Encoding real cells.")
    ncells = adata_sample.shape[0]
    indices = np.arange(ncells)
    counts = adata_sample.layers['counts']
    real_latent = []
    for i in range(0, ncells, batch_size):    
        # Get training data for current mini-batch        
        batch = indices[i:i + batch_size]
        train_data = torch.Tensor(adata_sample.layers['counts'][batch,:].toarray()).to(device)
        train_library = torch.Tensor(np.unique(adata_sample.obs['library_prep_encoded'])).long().repeat(train_data.shape[0]).to(device)
        train_source = torch.Tensor(np.unique(adata_sample.obs['cell_nuclei_encoded'])).long().repeat(train_data.shape[0]).to(device)
        train_study = torch.Tensor(np.unique(adata_sample.obs['study_encoded'])).long().repeat(train_data.shape[0]).to(device)
        train_donor = torch.Tensor(np.unique(adata_sample.obs['donor_encoded'])).long().repeat(train_data.shape[0]).to(device)
    
        px, rate, xhat, z, log_var, embeddings = model(x = train_data, donor = train_donor, study = train_study, library = train_library, source = train_source)
        real_latent.append(z.detach().cpu())
    real_latent = torch.concatenate(real_latent)
    real_label = torch.tensor(0).repeat(real_latent.shape[0])
    
    # Combine data
    latent = torch.vstack((doublet_latent, real_latent))
    label = torch.vstack((doublet_label.unsqueeze(-1), real_label.unsqueeze(-1)))
    
    # Test-train split
    n_samples = label.shape[0]
    indices = np.arange(n_samples)
    np.random.shuffle(indices) 
    n_train = int(n_samples * train_fraction)
    n_test = n_samples - n_train
    latent_train = latent[indices[:n_train],:]
    label_train = label[indices[:n_train],:]
    latent_test = latent[indices[n_train:n_samples],:]
    label_test = label[indices[n_train:n_samples],:]
    
    # Make classifier
    classifier = nn.Sequential(
        nn.Linear(real_latent.shape[1], n_hidden),
        nn.BatchNorm1d(n_hidden),
        nn.ReLU(),
        nn.Dropout(p = 0.1),
        nn.Linear(n_hidden, 1)
    ).to(device)
    
    # Train the classifier
    if verbose:
        print("Training classifier.")
    criterion = nn.BCEWithLogitsLoss()
    n_samples = label_train.shape[0]
    indices = np.arange(n_samples)
    n_test = latent_test.shape[0]
    test_indices = np.arange(n_test)
    best_acc = 0
    current_patience = 0
    optimizer = Adam(classifier.parameters(), lr=lr)
    for epoch in range(epochs):
        # Train
        np.random.shuffle(indices)
        classifier.train()
        for i in range(0, n_samples, batch_size):    
            # Zero the gradients
            optimizer.zero_grad()
            
            # Get training data for current mini-batch        
            batch = indices[i:i + batch_size]
            train_latent = latent_train[batch,:].to(device)
            train_label = label_train[batch,:].to(device)
        
            # Forward pass
            logits = classifier(train_latent)
        
            # Loss
            loss = criterion(logits, train_label.float())
        
            # Backward
            loss.backward()
        
            # Update
            optimizer.step()

        # Evaluate
        classifier.eval()
        hits = 0
        total = 0
        with torch.no_grad():
             for i in range(0, n_test, batch_size):    
                # Get test data for current mini-batch        
                batch = test_indices[i:i + batch_size]
                test_latent = latent_test[batch,:].to(device)
            
                # Forward pass
                test_class = torch.sigmoid(classifier(test_latent)).cpu().numpy()
        
                # Stats
                hits += np.sum(np.array(test_class >= 0.5, dtype = np.int64)[:,0] == label_test[batch,:].numpy()[:,0])
                total += batch.size

        # Evaluate accuracy
        acc = hits / total
        if acc >= best_acc:
            best_acc = acc
            best_state = copy.deepcopy(classifier.state_dict())
            current_patience = 0
        else:
            current_patience += 1

        # Early stopping
        if current_patience >= patience:
            break
        final_epoch = epoch + 1

    # Load best model
    classifier.load_state_dict(best_state)
    classifier.eval()
    hits = 0
    total = 0
    with torch.no_grad():
         for i in range(0, n_test, batch_size):    
            # Get test data for current mini-batch        
            batch = test_indices[i:i + batch_size]
            test_latent = latent_test[batch,:].to(device)
        
            # Forward pass
            test_class = torch.sigmoid(classifier(test_latent)).cpu().numpy()
    
            # Stats
            hits += np.sum(np.array(test_class >= 0.5, dtype = np.int64)[:,0] == label_test[batch,:].numpy()[:,0])
            total += batch.size

    # Evaluate accuracy
    acc = hits / total

    if verbose:
        print("Prediction accuracy = " + str(round(acc, 4)) + " achieved after " + str(final_epoch) + " epochs.")
    
    # Get probability for doublet in real cells
    if verbose:
        print("Calculating doublet probabilities.")
    n_real = real_latent.shape[0]
    indices = np.arange(n_real)
    classifier.eval()
    probs = []
    with torch.no_grad():
         for i in range(0, n_real, batch_size):    
            # Get training data for current mini-batch        
            batch = indices[i:i + batch_size]
            train_latent = real_latent[batch,:].to(device)
        
            # Forward pass
            probs.append(torch.sigmoid(classifier(train_latent)).cpu().numpy())
    probs = np.concatenate(probs)   

    if verbose: 
        print("Found " + str(round(np.sum(probs >= 0.9) / len(probs),3) * 100) + "% of real cells as doublets with high confidence")
        
    return probs

Function to get correct batch-sizes, because in some cases we might get a batch size of 1, and batchnorm1d can't handle that

In [64]:
def get_safe_batch_size(n_cells, train_fraction=0.8, doublet_ratio=1, max_batch_size=256):
    n_total = int(n_cells * (1 + doublet_ratio))
    n_train = int(n_total * train_fraction)

    # Try decreasing batch size until remainder is >= 2
    for bs in range(max_batch_size, 1, -1):
        if n_train % bs >= 2:
            return bs
    return 2  # fallback minimum

Run doublet detection using SOLO on own model for each sample

In [85]:
# Get samples for techniques that risk generating doublets (droplet based)
sample_ids = (
    adata.obs[(adata.obs["platform"].isin(["droplet"]))]["ic_id_sample"]
    .unique()
    .tolist()
)

# Collect results
results = []

for sample_id in sample_ids:
    print(f"Processing sample: {sample_id}")
    adata_sample = adata[adata.obs["ic_id_sample"] == sample_id, :].copy()

    # Check number of cells
    n_cells = adata_sample.n_obs

    if n_cells <= 1:
        print(f"There are only {n_cells} cells in {sample_id}, skipping this sample...")
        continue  # Skip to next sample

    # get safe number of batch sizes
    safe_batch_size = get_safe_batch_size(n_cells)

    # predict doublets
    probs = classify_doublets(
        model, adata_sample, batch_size=safe_batch_size, verbose=False
    )

    df = pd.DataFrame(
        {
            "barcode": adata_sample.obs.index,
            "batch_size": safe_batch_size,
            "doublet_probability": probs.ravel(),  # make 1 dimension
        }
    )

    df["ic_id_sample"] = sample_id
    results.append(df)

# Combine all results
final_df = pd.concat(results)

# Save to CSV
final_df.to_csv(os.path.join(files_dir, "doublet_probabilities.csv"), index=False)

print("Saved doublet probabilities to doublet_probabilities.csv")

Processing sample: ic_15_1_1
Processing sample: ic_15_2_2
Processing sample: ic_15_3_3
Processing sample: ic_15_4_4
Processing sample: ic_15_5_5
Processing sample: ic_15_6_6
Processing sample: ic_15_7_7
Processing sample: ic_15_8_8
Processing sample: ic_15_9_9
Processing sample: ic_15_10_10
Processing sample: ic_15_11_11
Processing sample: ic_15_12_12
Processing sample: ic_15_13_13
Processing sample: ic_15_14_14
Processing sample: ic_15_15_15
Processing sample: ic_15_16_16
Processing sample: ic_15_17_17
Processing sample: ic_15_18_18
Processing sample: ic_15_19_19
Processing sample: ic_15_20_20
Processing sample: ic_15_21_21
Processing sample: ic_15_22_22
Processing sample: ic_15_23_23
Processing sample: ic_15_24_24
Processing sample: ic_15_25_25
Processing sample: ic_15_26_26
Processing sample: ic_15_27_27
Processing sample: ic_15_28_28
Processing sample: ic_15_29_29
Processing sample: ic_15_30_30
Processing sample: ic_15_31_31
Processing sample: ic_15_32_32
Processing sample: ic_15_3

In [ ]:

# Load the CSV
doublet_df = pd.read_csv("doublet_probabilities.csv")

# Set index to cell_index for easy matching
doublet_df.set_index('cell_index', inplace=True)
